# Data Creation Playground

Full multi-depth pruning pipeline. G = Gemma-4 (Colab GPU), D = Gemini Flash Lite (cloud).
Source = `avreymi/reasoning-spectrum-qa` (1000 diverse QA across 6 reasoning families); each
question is assembled with its context and choices via `format_spectrum_question`, and answer
fields are never shown to G.

**Before running:** Enable GPU runtime → Runtime → Change runtime type → T4 GPU (or A100).

In [1]:
!git clone https://github.com/avrymi-asraf/reasoning-pruning.git
%cd reasoning-pruning
# Gemma 4 requires transformers from git main — not yet in a stable PyPI release
%pip install -q "git+https://github.com/huggingface/transformers.git" "accelerate>=0.34.0" "datasets>=4.8.5" "torchvision>=0.27.0" "pyyaml>=6.0.2"

Cloning into 'reasoning-pruning'...
remote: Enumerating objects: 278, done.
remote: Counting objects: 100% (278/278), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 278 (delta 102), reused 158 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (278/278), 761.65 KiB | 3.63 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/reasoning-pruning
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 162.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 15.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [2]:
import os
os.environ["HF_TOKEN"] = input("Enter your Hugging Face token: ")
os.environ["GEMINI_API_KEY"] = input("Enter your Gemini API key: ")

In [4]:
import os
import sys
sys.path.insert(0, "src")

from pathlib import Path
from dataclasses import replace

from reasoning_pruning.data_creation import (
    load_data_creation_config,
    load_questions,
    build_pt_dataset,
    build_rows_for_question,
    # format_context,
    # format_spectrum_question,
    # split_reasoning_units,
)
from reasoning_pruning.clients import TransformersGenerator, GeminiDecisionModel

print("Imports OK")

Imports OK


In [5]:
# Downloads ~5GB from Hub — takes 1-2 min on first run
generator = TransformersGenerator(
    source_model="avreymi/gemma-4-E2B-it-reasoning-pruning",
    generation_config={"max_new_tokens": 512, "temperature": 0.7, "do_sample": True},
)
print("G ready:", generator.source_model)

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# --- D-prompt iteration knob ---
# Swap PROMPT_VERSION to any file stem in prompts/, then re-run this cell and the
# build cells below. Compare the removal rate and the emitted input_x -> target_y rows.
#   "conservative-skip-v1"             — original, math-centric (mismatched for spectrum)
#   "conservative-skip-v2-general"     — "real reasoning" generalized to all 6 families
#   "conservative-skip-v2-family-aware"— per-family definition of a valid target
#   "balanced-skip-v1"                 — may remove multi-unit filler spans
PROMPT_VERSION = "conservative-skip-v2-general"

decision_model = GeminiDecisionModel(
    decision_model="gemini-flash-lite-latest",
    prompt_version=PROMPT_VERSION,
    prompts_dir="prompts",
)
print("D ready:", decision_model.decision_model, "| prompt:", PROMPT_VERSION)

In [ ]:
# Load config from YAML, then override depth/limit for quick playground runs.
# Source is avreymi/reasoning-spectrum-qa — questions are pulled straight from it,
# with context + choices already assembled by format_spectrum_question.
config = load_data_creation_config(Path("configs/data/dataset_builder_spectrum_gemma4.yaml"))
config = replace(config, max_pruning_depth=4, max_examples_per_question=4, source_limit=8)

questions = load_questions(config, hf_token=os.environ.get("HF_TOKEN"))

print(f"Config: max_depth={config.max_pruning_depth}, G={config.generator['model_id']}")
print(f"Source: {config.source_dataset} (split={config.source_split})")
print(f"Questions: {len(questions)}")
print("\n--- Example assembled question (context + choices, no answer) ---\n")
print(questions[0])

In [9]:
def show_rows(rows: list[dict]) -> None:
    if not rows:
        print("No rows generated — D found no safe removals at any depth")
        return
    for row in rows:
        sep = "=" * 70
        print(f"\n{sep}")
        print(f"  Depth {row['pruning_depth']}")
        print(f"{sep}")
        print(f"\n[GENERATED UNITS]")
        for i, u in enumerate(row["generated_units"]):
            marker = "  ✗" if row["metadata"]["removed_start_index"] <= i <= row["metadata"]["removed_end_index"] else "   "
            print(f"{marker} {i}: {u}")
        print(f"\n[REMOVED] indices {row['metadata']['removed_start_index']}–{row['metadata']['removed_end_index']}")
        print(f"  Reason: {row['metadata']['decision_reason']}")
        print(f"\n[INPUT_X]")
        for line in row["input_x"].splitlines():
            print(f"  {line}")
        print(f"\n[TARGET_Y]")
        print(f"  {row['target_y']}")
    print(f"\n  → {len(rows)} training row(s) from this question")

In [10]:
# Run on a single question to inspect each depth in detail
rows = build_rows_for_question(
    question=questions[1],
    generator=generator,
    decision_model=decision_model,
    config=config,
)
show_rows(rows)


  Depth 0

[GENERATED UNITS]
    0: Fact 1: There are seven objects arranged in a fixed order.
    1: Objects: Owl (O), Crow (C), Falcon (F), Cardinal (Ca), Hummingbird (H), Quail (Q), Hawk (Ha).
    2: Positions: 1, 2, 3, 4, 5, 6, 7 (from left to right).
    3: Fact 2: The falcon is the second from the right.
    4: Computation 1: Position of F is $7 - 2 + 1 = 6$.
    5: Arrangement: \_, \_, \_, \_, \_, F, \_
    6: Fact 3: The owl is the third from the left.
    7: Computation 2: Position of O is 3.
    8: Arrangement: \_, \_, O, \_, \_, F, \_
    9: Fact 4: The falcon is to the left of the crow.
    10: Computation 3: Position of C is $> $ Position of F (6). This is impossible since F is at position 6 and there is only one spot to the right (position 7). Re-reading Fact 2: "The falcon is the second from the right." Positions are 1, 2, 3, 4, 5, 6, 7. Second from right is position 6.
    11: If F is at 6, C must be at 7.
    12: Arrangement: \_, \_, O, \_, \_, F, C.
  ✗ 13: Fact 5: T

In [ ]:
# Run all questions and get a summary
all_rows = build_pt_dataset(
    questions=questions,
    generator=generator,
    decision_model=decision_model,
    config=config,
)

print(f"Total rows: {len(all_rows)}")
print(f"Rows per question: {len(all_rows) / len(questions):.1f} avg")
print(f"Depths seen: {sorted(set(r['pruning_depth'] for r in all_rows))}")
print()
for row in all_rows:
    q_preview = row["question"][:60]
    removed = " | ".join(row["metadata"]["removed_span"])
    print(f"[d={row['pruning_depth']}] {q_preview}...")
    print(f"  removed: {removed[:80]}")
    print(f"  target : {row['target_y'][:80]}")